In [26]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# ==========================================
# 0. GLOBAL PLOTTING & ENVIRONMENT SETUP
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

#pl.Config.set_tbl_rows(-1)             # shows all table rows globably
#pl.Config.set_tbl_cols(-1)             # shows all table columns globably
#pl.Config.set_fmt_str_lengths(100)     # increase character count visible globably
#pl.Config.restore_defaults()           # reset to default configuration

# Get current working directory (e.g., .\notebooks)
cwd = os.getcwd()

# Get parent folder to access (.\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_EPTI_data = os.path.join(parent, "Data", "EPTI_CET_data.parquet")

print(f"Loading dataset from: {file_path_EPTI_data}")
df = pl.read_parquet(file_path_EPTI_data).with_columns(pl.col(["Price_profile", "Price_level", "Region",
                                                                "Municipality", "Group_Phase2", "Control_Price_Phase2"]).cast(pl.Categorical))

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\EPTI_CET_data.parquet


In [27]:
# Check for missing values
def what_percentage_missing(df):
    missing_percentage = (
        df.select((pl.all().is_null().sum() / pl.len()) * 100)
        .transpose(include_header=True, header_name="column", column_names=["pct_missing"])
        .sort("pct_missing", descending=True)
    )

    print("Percentage of missing values in each column:")

    with pl.Config(tbl_rows=-1):
        print(missing_percentage)

what_percentage_missing(df)

Percentage of missing values in each column:
shape: (18, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 88.006897   │
│ Price_level              ┆ 88.006897   │
│ Experiment_price_NOK_kWh ┆ 88.006897   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [28]:
# Create a boolean expression checking for "Control"
is_control = pl.col("Group_Phase2").cast(pl.String).str.contains("Control")

# Summary DataFrame with counts and percentages
summary_df = (
    df.with_columns(
        pl.when(is_control)
        .then(pl.lit("Control"))
        .otherwise(pl.lit("Experiment"))
        .alias("Cohort")
    )
    .group_by("Cohort")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct")
    )
)

print(summary_df)

shape: (2, 3)
┌────────────┬─────────┬───────┐
│ Cohort     ┆ count   ┆ pct   │
│ ---        ┆ ---     ┆ ---   │
│ str        ┆ u32     ┆ f64   │
╞════════════╪═════════╪═══════╡
│ Experiment ┆ 8638752 ┆ 70.93 │
│ Control    ┆ 3541248 ┆ 29.07 │
└────────────┴─────────┴───────┘


In [29]:
# Subset containing Ber_Control, Bo_Control, Control, Os_Control, Trom_Control, Trond_Control
df_control = df.filter(is_control)

# Subset containing everything else (Ber_1, Bo_1, H1, H2, NA, Os_1..6, Trom_1..2, Trond_1)
df_experiment = df.filter(~is_control)

In [30]:
# just the dataframe with 
what_percentage_missing(df_experiment)

Percentage of missing values in each column:
shape: (18, 2)
┌──────────────────────────┬─────────────┐
│ column                   ┆ pct_missing │
│ ---                      ┆ ---         │
│ str                      ┆ f64         │
╞══════════════════════════╪═════════════╡
│ Price_profile            ┆ 83.090613   │
│ Price_level              ┆ 83.090613   │
│ Experiment_price_NOK_kWh ┆ 83.090613   │
│ ID                       ┆ 0.0         │
│ From                     ┆ 0.0         │
│ Date                     ┆ 0.0         │
│ Day_Of_Week              ┆ 0.0         │
│ Is_Weekend               ┆ 0.0         │
│ Hour                     ┆ 0.0         │
│ Demand_kWh               ┆ 0.0         │
│ Temperature              ┆ 0.0         │
│ Temperature24            ┆ 0.0         │
│ Temperature48            ┆ 0.0         │
│ Temperature72            ┆ 0.0         │
│ Municipality             ┆ 0.0         │
│ Region                   ┆ 0.0         │
│ Control_Price_Phase2     ┆ 0.0     

In [ ]:
# 100% missing - as expected
#what_percentage_missing(df_control)

In [35]:
# How many and what proportion of those experiment days fall specifically on weekends
total_rows = df.height

stats = (
    df_experiment
    .filter(
        (pl.col("Is_Weekend") == 1) & 
        (pl.col("Price_level").is_not_null())
    )
    .select(
        Match_Count = pl.len()
    )
    .with_columns(
        Percentage = (pl.col("Match_Count") / total_rows) * 100
    )
)

print(total_rows)
print(stats)

12180000
shape: (1, 2)
┌─────────────┬────────────┐
│ Match_Count ┆ Percentage │
│ ---         ┆ ---        │
│ u32         ┆ f64        │
╞═════════════╪════════════╡
│ 9813        ┆ 0.080567   │
└─────────────┴────────────┘


In [31]:
# Check for duplicate rows
duplicate_rows = df.is_duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


In [32]:
# Check data types
print("Data types of each column:")
schema_df = pl.DataFrame({
    "Column": df.columns,
    "Data Type": [str(dtype) for dtype in df.dtypes]})

with pl.Config(tbl_rows=-1):
    print(schema_df)

Data types of each column:
shape: (18, 2)
┌──────────────────────────┬─────────────────────────────────┐
│ Column                   ┆ Data Type                       │
│ ---                      ┆ ---                             │
│ str                      ┆ str                             │
╞══════════════════════════╪═════════════════════════════════╡
│ ID                       ┆ String                          │
│ From                     ┆ Datetime(time_unit='ms', time_… │
│ Date                     ┆ Date                            │
│ Day_Of_Week              ┆ Int32                           │
│ Is_Weekend               ┆ Int32                           │
│ Hour                     ┆ Int32                           │
│ Demand_kWh               ┆ Float64                         │
│ Price_profile            ┆ Categorical                     │
│ Price_level              ┆ Categorical                     │
│ Experiment_price_NOK_kWh ┆ Float64                         │
│ Temperature

In [ ]:
# Calculate the target label (Y)
